# Définition des constantes

In [ ]:
import sys, platform
from pathlib import Path
import torch

ROOT = Path.cwd().parent

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

DATA = ROOT / "Data"
SRC = ROOT / "SRC"
SPLIT = DATA / "SPLIT"


# Importation du jeu de données

In [2]:
import pandas as pd

columns_name = ['TARGET', 'id', 'date', '??', 'user', 'tweet']

df = pd.read_csv(DATA / "training.1600000.processed.noemoticon.csv", encoding='ISO-8859-1', names=columns_name)
print(df.head())

   TARGET          id                          date        ??  \
0       0  1467810369  Mon Apr 06 22:19:45 PDT 2009  NO_QUERY   
1       0  1467810672  Mon Apr 06 22:19:49 PDT 2009  NO_QUERY   
2       0  1467810917  Mon Apr 06 22:19:53 PDT 2009  NO_QUERY   
3       0  1467811184  Mon Apr 06 22:19:57 PDT 2009  NO_QUERY   
4       0  1467811193  Mon Apr 06 22:19:57 PDT 2009  NO_QUERY   

              user                                              tweet  
0  _TheSpecialOne_  @switchfoot http://twitpic.com/2y1zl - Awww, t...  
1    scotthamilton  is upset that he can't update his Facebook by ...  
2         mattycus  @Kenichan I dived many times for the ball. Man...  
3          ElleCTF    my whole body feels itchy and like its on fire   
4           Karoli  @nationwideclass no, it's not behaving at all....  


# Train Test Split

In [11]:
from sklearn.model_selection import train_test_split

df["TARGET_BINARY"] = df["TARGET"].map({0: 0, 4: 1})
dftrainval, dftest = train_test_split(df[["TARGET_BINARY", "tweet"]], test_size=50_000, random_state=42, stratify=df["TARGET_BINARY"])

dftrain, dfval = train_test_split(dftrainval, test_size=25_000, random_state=42, stratify=dftrainval["TARGET_BINARY"])


## Sauvegarde des splits

In [ ]:

#TRAIN
dftrain.to_csv(SPLIT / "train.csv", index=False, encoding="utf-8")

#VALIDATION
dfval.to_csv(SPLIT / "val.csv", index=False, encoding="utf-8")

#TEST
dftest.to_csv(SPLIT / "test.csv", index=False, encoding="utf-8")

# PIPELINE 1 (LSTM)

## Pré-traitement

### Nettoyage

In [21]:
from SRC.preprocessing import clean_tweet_LSTM

df_train_LSTM = pd.read_csv(SPLIT / "train.csv", encoding='utf-8')
df_train_LSTM = df_train_LSTM.sample(
    n=300_000, random_state=42
).reset_index(drop=True)


df_val_LSTM = pd.read_csv(SPLIT / "val.csv", encoding='utf-8')
df_test_LSTM = pd.read_csv(SPLIT / "test.csv", encoding='utf-8')

df_train_LSTM["tweet_net"] = df_train_LSTM["tweet"].apply(clean_tweet_LSTM)
df_val_LSTM["tweet_net"] = df_val_LSTM["tweet"].apply(clean_tweet_LSTM)
df_test_LSTM["tweet_net"] = df_test_LSTM["tweet"].apply(clean_tweet_LSTM)

print(df_train_LSTM[["tweet", "tweet_net"]].head())

                                               tweet  \
0  Watching Ramsay's Kitchen Nightmares now  Ew c...   
1  On my way too the beachh with thee bitchess  a...   
2  My hand is swollen, bruised and all  the shit ...   
3  @MrBillyBones although it would be the highlig...   
4             Going to the beach with Cody and Jake    

                                           tweet_net  
0  watching ramsays kitchen nightmares now ew coc...  
1  on my way too the beachh with thee bitchess ah...  
2  my hand is swollen bruised and all the shit hu...  
3  although it would be the highlight of the summ...  
4              going to the beach with cody and jake  


### Embedding (avec Glove)

In [22]:
if not(Path(DATA / "Embedding" / "emb_matrix_300k.npy").exists()):

    import numpy as np
    from collections import Counter

    MAX_VOCAB = 30_000
    EMB_DIM = 200
    GLOVE_PATH = DATA / "Embedding" / "glove.twitter.27B.200d.txt"

    # --- Vocabulaire à partir du train nettoyé ---
    counter = Counter()
    for txt in df_train_LSTM["tweet_net"]:
        counter.update(str(txt).split())

    # index 0 = <pad>, index 1 = <unk>
    itos = ["<pad>", "<unk>"] + [w for w, _ in counter.most_common(MAX_VOCAB - 2)]
    stoi = {w: i for i, w in enumerate(itos)}
    print(f"Vocabulaire : {len(itos):,} tokens")

    # --- Matrice d'embeddings ---
    rng = np.random.default_rng(42)
    emb_matrix = rng.normal(0, 0.1, (len(itos), EMB_DIM)).astype(np.float32)
    emb_matrix[0] = 0.0  # <pad> à zéro

    found = 0
    with open(GLOVE_PATH, encoding="utf-8") as f:
        for line in f:
            parts = line.rstrip().split(" ")
            word = parts[0]
            idx = stoi.get(word)
            if idx is not None:
                emb_matrix[idx] = np.asarray(parts[1:], dtype=np.float32)
                found += 1

    print(f"Couverture GloVe : {found:,}/{len(itos):,} ({found/len(itos):.1%})")

    np.save(DATA / "Embedding" / "emb_matrix_300k.npy", emb_matrix)
    lengths = df_train_LSTM["tweet_net"].str.split().str.len()
    print(lengths.describe())
    print(f"p95 : {lengths.quantile(0.95):.0f} | p99 : {lengths.quantile(0.99):.0f}")
else:
    emb_matrix = np.load(DATA / "Embedding" / "emb_matrix_300k.npy")

## Entrainement

### tweets => Matrices

In [23]:
import numpy as np
import torch

MAX_LEN = 40
PAD, UNK = 0, 1

def textes_vers_matrice(series_textes):
    """Convertit une colonne de textes en matrice (n_tweets, 40)."""
    matrice = np.zeros((len(series_textes), MAX_LEN), dtype=np.int64)
    for i, texte in enumerate(series_textes):
        mots = str(texte).split()[:MAX_LEN]
        for j, mot in enumerate(mots):
            matrice[i, j] = stoi.get(mot, UNK)
    return matrice

# Conversion en tenseurs
X_train = torch.tensor(textes_vers_matrice(df_train_LSTM["tweet_net"]))
y_train = torch.tensor(df_train_LSTM["TARGET_BINARY"].values, dtype=torch.float32)

X_val = torch.tensor(textes_vers_matrice(df_val_LSTM["tweet_net"]))
y_val = torch.tensor(df_val_LSTM["TARGET_BINARY"].values, dtype=torch.float32)

X_test = torch.tensor(textes_vers_matrice(df_test_LSTM["tweet_net"]))
y_test = torch.tensor(df_test_LSTM["TARGET_BINARY"].values, dtype=torch.float32)

print(X_train.shape, y_train.shape)

torch.Size([300000, 40]) torch.Size([300000])


### Modèle

In [24]:
import torch.nn as nn

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(DEVICE)

class LSTMSentiment(nn.Module):
    def __init__(self, emb_matrix):
        super().__init__()
        # 1. Embedding : indice -> vecteur GloVe de dim 200
        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(emb_matrix), freeze=False, padding_idx=PAD
        )
        # 2. LSTM bidirectionnel : lit le tweet dans les deux sens
        self.lstm = nn.LSTM(200, 128, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(0.3)
        # 3. Sortie : 256 -> 1 score
        self.fc = nn.Linear(256, 1)

    def forward(self, x):
        emb = self.embedding(x)              # (batch, 40, 200)
        sorties, _ = self.lstm(emb)          # (batch, 40, 256)
        moyenne = sorties.mean(dim=1)        # (batch, 256)
        return self.fc(self.dropout(moyenne)).squeeze(1)

model = LSTMSentiment(emb_matrix).to(DEVICE)

cpu


### boucle d'entrainement

In [ ]:
from sklearn.metrics import accuracy_score, f1_score
import time, copy

best_f1, best_state, patience = 0, None, 0
BATCH = 256
EPOCHS = 5

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def predire(X):
    model.eval()
    predictions = []
    with torch.no_grad():
        for i in range(0, len(X), 512):
            logits = model(X[i:i+512].to(DEVICE))
            predictions += (torch.sigmoid(logits) > 0.5).long().cpu().tolist()
    return predictions


for epoch in range(1, EPOCHS + 1):
    model.train()
    perm = torch.randperm(len(X_train))     # mélange à chaque epoch
    perte_totale, t0 = 0, time.time()

    for i in range(0, len(X_train), BATCH):
        idx = perm[i:i+BATCH]
        xb, yb = X_train[idx].to(DEVICE), y_train[idx].to(DEVICE)

        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        perte_totale += loss.item()
    preds = predire(X_val)
    f1 = f1_score(y_val, preds, average="macro")
    acc = accuracy_score(y_val, preds)
    print(f"Epoch {epoch} | loss {perte_totale/(len(X_train)//BATCH):.4f} | "
          f"val acc {acc:.4f} | val F1 {f1:.4f} | {time.time()-t0:.0f}s")

    if f1 > best_f1:
        best_f1, best_state, patience = f1, copy.deepcopy(model.state_dict()), 0
    else:
        patience += 1
        if patience >= 2:
            print(f"Early stopping — meilleur F1 val : {best_f1:.4f}")
            break

if best_state is not None:
    model.load_state_dict(best_state)
    print(f"Modèle restauré — F1 val : {best_f1:.4f}")

Epoch 1 | loss 0.4474 | val acc 0.8114 | val F1 0.8113 | 156s
Epoch 2 | loss 0.3889 | val acc 0.8177 | val F1 0.8177 | 150s
Epoch 3 | loss 0.3539 | val acc 0.8190 | val F1 0.8189 | 149s
Epoch 4 | loss 0.3176 | val acc 0.8134 | val F1 0.8134 | 168s
Epoch 5 | loss 0.2788 | val acc 0.8070 | val F1 0.8070 | 150s


## Tests

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

preds_test = predire(X_test)
print(confusion_matrix(y_test, preds_test))
print(classification_report(y_test, preds_test, target_names=["négatif", "positif"]))

(ROOT / "Models").mkdir(exist_ok=True)

torch.save({
    "state_dict": model.state_dict(),
    "itos": itos,
    "stoi": stoi,
    "max_len": MAX_LEN,
}, ROOT / "Models" / "lstm_baseline.pt")

TEST — accuracy 0.8056 | F1 macro 0.8055


# PIPELINE 2 (SETFIT)

## Pré-traitement

### Nettoyage

In [ ]:
from SRC.preprocessing import clean_tweet_SETFIT

df_train_SETFIT = pd.read_csv(SPLIT / "train.csv", encoding='utf-8')
df_train_SETFIT["tweet_net"] = df_train_SETFIT["tweet"].apply(clean_tweet_SETFIT)

print(df_train_SETFIT[["tweet", "tweet_net"]].head())

                                               tweet  \
0  @switchfoot http://twitpic.com/2y1zl - Awww, t...   
1  is upset that he can't update his Facebook by ...   
2  @Kenichan I dived many times for the ball. Man...   
3    my whole body feels itchy and like its on fire    
4  @nationwideclass no, it's not behaving at all....   

                                           tweet_net  
0  @user http - Aww, that's a bummer. You shoulda...  
1  is upset that he can't update his Facebook by ...  
2  @user I dived many times for the ball. Managed...  
3     my whole body feels itchy and like its on fire  
4  @user no, it's not behaving at all. i'm mad. w...  


## Entrainement

## Tests